<a href="https://colab.research.google.com/github/Mansi114114/Oral_Diseases_Prediction/blob/main/oral_diseases_with_fine_tuning_transfer_learning_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_path = '/content/drive/MyDrive/Teeth_Dataset'


In [ ]:
# Update the paths to the correct dataset location
train_dir = '/content/drive/MyDrive/Teeth_Dataset/Training'
validation_dir = '/content/drive/MyDrive/Teeth_Dataset/Validation'

# Import the necessary function
from tensorflow.keras.utils import image_dataset_from_directory

# Load the training dataset
train_dataset = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(128, 128),
    batch_size=32
)

# Load the validation dataset
val_dataset = image_dataset_from_directory(
    validation_dir,
    labels='inferred',
    label_mode='int',
    image_size=(128, 128),
    batch_size=32
)

# Get the class names
class_names = train_dataset.class_names
print(f"Class names: {class_names}")

Found 3087 files belonging to 7 classes.
Found 1028 files belonging to 7 classes.
Class names: ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Teeth_Dataset-20251125T195834Z-1-001.zip'
extract_path = '/content/drive/MyDrive/'

if not os.path.exists(zip_path):
    print(f"Error: Zip file not found at {zip_path}")
else:
    print(f"Unzipping {zip_path} to {extract_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Unzipping complete. Now checking contents:")
    # Verify that the directory structure is created
    if os.path.exists(os.path.join(extract_path, 'Teeth_Dataset')):
        print(f"'Teeth_Dataset' directory created at {os.path.join(extract_path, 'Teeth_Dataset')}")
        print("Listing contents of the new 'Teeth_Dataset' directory:")
        for item in os.listdir(os.path.join(extract_path, 'Teeth_Dataset')):
            print(f'- {item}')
    else:
        print("Warning: 'Teeth_Dataset' directory was not found after unzipping. Please check zip file contents.")


Unzipping /content/drive/MyDrive/Teeth_Dataset-20251125T195834Z-1-001.zip to /content/drive/MyDrive/
Unzipping complete. Now checking contents:
'Teeth_Dataset' directory created at /content/drive/MyDrive/Teeth_Dataset
Listing contents of the new 'Teeth_Dataset' directory:
- Validation
- Testing
- Training


In [ ]:
import os

print('Listing contents of /content/drive/MyDrive:')
for item in os.listdir('/content/drive/MyDrive'):
    print(f'- {item}')

print('\nListing contents of /content/drive/MyDrive/Teeth_Dataset (if it exists):')
try:
    for item in os.listdir('/content/drive/MyDrive/Teeth_Dataset'):
        print(f'- {item}')
except FileNotFoundError:
    print('The directory /content/drive/MyDrive/Teeth_Dataset does not exist or is not accessible.')


Listing contents of /content/drive/MyDrive:
- 12201012024_Mythri Jadon.pdf
- Classroom
- 12201012024_Experiment-03 (1).pdf
- 12201012024_Experiment-03.pdf
- Colab Notebooks
- WhatsApp Image 2025-04-06 at 5.25.55 PM.jpeg
- Avira
- IDS Project Lab File.gdoc
- EVS Report 2025.gdoc
- payment for cvdl.pdf
- Undertaking Form for others.pdf
- Undertaking Form for others.docx
- Photo
- Mythri Jadon _ Krishna hostel 
- 12201012024_MythriJadon_CSE2_IoT_Phase2.mp4
- IMG-20251123-WA0066.jpg
- Teeth_Dataset
- Teeth_Dataset-20251125T195834Z-1-001.zip

Listing contents of /content/drive/MyDrive/Teeth_Dataset (if it exists):
- Validation
- Testing
- Training


In [ ]:
import tensorflow as tf

# Define a preprocessing function for normalization
def preprocess_image(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32) / 255.0
    return image, label

# Apply preprocessing to both datasets
train_dataset = train_dataset.map(preprocess_image)
val_dataset = val_dataset.map(preprocess_image)

# Configure the datasets for performance
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

print("Preprocessing and data pipeline configuration complete.")

Preprocessing and data pipeline configuration complete.


In [ ]:
# Define the image size
image_size = (128, 128)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import Model

# Determine the number of classes from the class_names variable
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

# Define the model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=image_size + (3,)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.summary()

Number of classes: 7


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,543 (12.61 MB)

 Trainable params: 3,305,543 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

# Configure the model for training
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

print("Model compilation complete.")

Model compilation complete.


In [ ]:
from tensorflow.keras.applications import VGG16

# Choose a pre-trained model (e.g., VGG16)
# We will load weights pre-trained on ImageNet, and exclude the top classification layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

print("VGG16 pre-trained model loaded.")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
VGG16 pre-trained model loaded.


In [ ]:
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model

# Freeze the layers of the base model so they are not trained
base_model.trainable = False

# Add new classification layers on top of the base model
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

# Create the new model
transfer_model = Model(inputs=base_model.input, outputs=predictions)

transfer_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,764,295 (60.14 MB)

 Trainable params: 1,049,607 (4.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
import tensorflow as tf

# Compile the new model
transfer_model.compile(optimizer='adam',
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                       metrics=['accuracy'])

print("Transfer learning model compilation complete.")

Transfer learning model compilation complete.


In [ ]:
epochs_transfer = 10 # You can adjust this number

history_transfer = transfer_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs_transfer
)

print("Transfer learning model training complete.")

Epoch 1/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 719s 7s/step - accuracy: 0.5120 - loss: 1.3023 - val_accuracy: 0.6761 - val_loss: 1.0442
Epoch 2/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 711s 7s/step - accuracy: 0.5773 - loss: 1.1209 - val_accuracy: 0.7403 - val_loss: 0.8443
Epoch 3/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 712s 7s/step - accuracy: 0.6720 - loss: 0.8856 - val_accuracy: 0.7977 - val_loss: 0.6850
Epoch 4/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 709s 7s/step - accuracy: 0.7300 - loss: 0.7378 - val_accuracy: 0.8045 - val_loss: 0.6167
Epoch 5/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 740s 7s/step - accuracy: 0.7551 - loss: 0.6840 - val_accuracy: 0.8492 - val_loss: 0.5394
Epoch 6/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 764s 8s/step - accuracy: 0.7964 - loss: 0.5754 - val_accuracy: 0.8551 - val_loss: 0.4788
Epoch 7/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 702s 7s/step - accuracy: 0.8284 - loss: 0.4818 - val_accuracy: 0.8580 - val_loss: 0.4233
Epoch 8/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 707s 7s/step - accuracy: 0.8268 - loss: 0.4676 - val_accuracy: 0.8735 - v

In [ ]:
import time
while True:
    time.sleep(60)
    print("Keeping session alive...")

Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...
Keeping session alive...


KeyboardInterrupt: 

In [ ]:
# Evaluate the transfer learning model
loss_transfer, accuracy_transfer = transfer_model.evaluate(val_dataset)

print(f"Transfer Learning Model Validation Loss: {loss_transfer}")
print(f"Transfer Learning Model Validation Accuracy: {accuracy_transfer}")

33/33 ━━━━━━━━━━━━━━━━━━━━ 177s 5s/step - accuracy: 0.8795 - loss: 0.3637
Transfer Learning Model Validation Loss: 0.36258018016815186
Transfer Learning Model Validation Accuracy: 0.8803501725196838


In [ ]:
# Save the transfer learning model
transfer_model.save('teeth_disease_transfer_model.h5')

print("Transfer learning model saved as 'teeth_disease_transfer_model.h5'")

Transfer learning model saved as 'teeth_disease_transfer_model.h5'


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_disease(img_path):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(128, 128)) # Ensure this matches your model's input size
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Create a batch
    img_array = img_array / 255.0  # Normalize the image

    # Make a prediction
    predictions = transfer_model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    # Get the predicted class index
    predicted_class_index = np.argmax(score)

    # Get the predicted class name
    predicted_class_name = class_names[predicted_class_index]

    return predicted_class_name, score[predicted_class_index]

    print("Prediction function defined.")

    # Prepare additional information for each disease class
disease_info = {
    'CaS': {
        'precautions': [
            ' Avoid acidic/spicy foods, maintain oral hygiene, reduce stress, use soft-bristled toothbrush'
        ],
        'warnings': [
            'Painful round ulcers with white/yellow center, red border, pain when eating.',
       ],
        'additional_info': 'Non-contagious; usually heal in 1–2 weeks; cause often unknown but linked to immunity and diet'
    },
    'CoS': {
        'precautions': ['Avoid kissing/sharing utensils with infected person, use sunblock on lips, manage stress'],
        'warnings': ['Tingling/burning around lips, small fluid-filled blisters, itching'],
        'additional_info': 'Caused by herpes simplex virus; highly contagious during outbreak'
    },
    'Gum': {
        'precautions': ['Brush twice daily, floss daily, avoid smoking, regular dental cleaning'],
        'warnings': ['Swollen/red gums, bleeding during brushing, bad breath, receding gums'],
        'additional_info': 'Caused by plaque buildup; can lead to tooth loss if untreated; linked to heart disease risk'
    },
    'MC': {
        'precautions': [f'Avoid tobacco & alcohol, maintain balanced diet, use protective lip balm against sun'],
        'warnings': ['Non-healing ulcers, lumps, pain, difficulty speaking/swallowing'],
        'additional_info': 'May refer to cancers in tongue, gums, lips, and palate; overlaps with OC but classification may be location-specific.'
    },
    'OC': {
        'precautions': ['Avoid tobacco (smoking & chewing), limit alcohol, eat antioxidant-rich foods, get regular dental checkups'],
        'warnings': ['Persistent mouth sores, lump in mouth, difficulty swallowing, unexplained bleeding'],
        'additional_info': 'Can be life-threatening if untreated; early detection improves survival.'
    },
    'OLP': {
        'precautions': ['Maintain good oral hygiene, avoid tobacco & alcohol, manage stress, avoid spicy/acidic foods'],
        'warnings': ['White lace-like patches, burning sensation, redness, painful ulcers'],
        'additional_info': 'Chronic inflammatory condition affecting oral mucosa; not contagious but may be linked to autoimmune response.'
    },
    'OT': {
        'precautions': ['Maintain oral hygiene, rinse mouth after inhaler use, control diabetes, limit sugary foods'],
        'warnings': ['White creamy patches inside mouth, soreness, loss of taste'],
        'additional_info': 'Caused by Candida fungal overgrowth; common in infants, elderly, and people with weak immunity.'
    }
}



In [ ]:
# Create a dictionary to map short class names to full disease names
disease_name_mapping = {
    'CaS': 'Canker Sores (Aphthous Ulcers)',
    'CoS': 'Cold Sores (Herpes Labialis)',
    'Gum': 'Gum Disease (Periodontal Disease)',
    'MC': 'Mouth Cancer',
    'OC': 'Oral Cancer',
    'OLP': 'Oral Lichen Planus',
    'OT': 'Oral Thrush'
}

print("Disease name mapping dictionary created.")

Disease name mapping dictionary created.


### Fine-tuning the transfer learning model

In [ ]:
# Unfreeze some layers of the base model for fine-tuning
# It's common to unfreeze the later layers
fine_tune_at = -4 # Unfreeze the last 4 layers of the VGG16 base model

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

# Re-compile the model after unfreezing layers
transfer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Use a lower learning rate for fine-tuning
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                       metrics=['accuracy'])

transfer_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,764,295 (60.14 MB)

 Trainable params: 8,129,031 (31.01 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
  layers.RandomContrast(0.1),
])

# Apply data augmentation to the training dataset
train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))

print("Data augmentation applied to the training dataset.")

Data augmentation applied to the training dataset.


Now, train the fine-tuned model for a few more epochs.

In [ ]:
# Continue training the model with fine-tuning
fine_tune_epochs = 10 # You can adjust this number
total_epochs = epochs_transfer + fine_tune_epochs

history_fine_tune = transfer_model.fit(
    train_dataset,
    epochs=total_epochs,
    initial_epoch=history_transfer.epoch[-1], # Start from the last epoch of the previous training
    validation_data=val_dataset
)

print("Fine-tuning complete.")

Epoch 10/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 828s 9s/step - accuracy: 0.7193 - loss: 0.7912 - val_accuracy: 0.9037 - val_loss: 0.2999
Epoch 11/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 822s 8s/step - accuracy: 0.7773 - loss: 0.6335 - val_accuracy: 0.9222 - val_loss: 0.2649
Epoch 12/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 823s 9s/step - accuracy: 0.7894 - loss: 0.5832 - val_accuracy: 0.9232 - val_loss: 0.2434
Epoch 13/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 823s 9s/step - accuracy: 0.8208 - loss: 0.4837 - val_accuracy: 0.9309 - val_loss: 0.2189
Epoch 14/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 822s 8s/step - accuracy: 0.8429 - loss: 0.4621 - val_accuracy: 0.9446 - val_loss: 0.1801
Epoch 15/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 847s 9s/step - accuracy: 0.8433 - loss: 0.4538 - val_accuracy: 0.9465 - val_loss: 0.1821
Epoch 16/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 822s 8s/step - accuracy: 0.8388 - loss: 0.4464 - val_accuracy: 0.9553 - val_loss: 0.1508
Epoch 17/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 829s 9s/step - accuracy: 0.8651 - loss: 0.3891 - val_accuracy: 0.

After fine-tuning, evaluate the model's performance.

In [ ]:
# Evaluate the fine-tuned model
loss_fine_tune, accuracy_fine_tune = transfer_model.evaluate(val_dataset)

print(f"Fine-tuned Model Validation Loss: {loss_fine_tune}")
print(f"Fine-tuned Model Validation Accuracy: {accuracy_fine_tune}")

33/33 ━━━━━━━━━━━━━━━━━━━━ 177s 5s/step - accuracy: 0.9697 - loss: 0.1090
Fine-tuned Model Validation Loss: 0.10165348649024963
Fine-tuned Model Validation Accuracy: 0.9737353920936584


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras import Model

# Ensure num_classes is defined (assuming class_names from earlier execution is available)
if 'num_classes' not in locals():
    print("num_classes not found, attempting to re-derive from class_names.")
    if 'class_names' in locals():
        num_classes = len(class_names)
        print(f"Re-derived num_classes: {num_classes}")
    else:
        raise ValueError("class_names not found. Please re-run dataset loading cell (e.g., 8warN7Aobre_) first.")

# Load the base VGG16 model again (excluding the top classification layer)
# Ensure image_size is defined (assuming it's from earlier execution)
if 'image_size' not in locals():
    print("image_size not found, setting to default (128, 128).")
    image_size = (128, 128)

base_model = VGG16(weights='imagenet', include_top=False, input_shape=image_size + (3,))

# Freeze the layers of the base model initially
base_model.trainable = False

# Re-create the new classification layers on top of the base model
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

# Re-create the transfer_model
transfer_model = Model(inputs=base_model.input, outputs=predictions)

# Re-compile the model for fine-tuning (matching the settings in a5aea19f)
# Unfreeze some layers of the base model for fine-tuning
fine_tune_at = -4 # Unfreeze the last 4 layers of the VGG16 base model

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

transfer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Use a lower learning rate for fine-tuning
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                       metrics=['accuracy'])

print("transfer_model architecture re-created and re-compiled for fine-tuning.")
print("Please re-run the fine-tuning cell (bb7b916) next to restore the trained weights, then proceed with evaluation.")

transfer_model architecture re-created and re-compiled for fine-tuning.
Please re-run the fine-tuning cell (bb7b916) next to restore the trained weights, then proceed with evaluation.


### Restarting the session requires re-running these cells to define essential variables:

Before running the Gradio interface, please execute the following cells to ensure all necessary data and mappings are loaded into memory:

1.  **Cell `8warN7Aobre_`**: Load the training and validation datasets, and define `class_names`.
2.  **Cell `6c89ed5a`**: Preprocess the datasets.
3.  **Cell `SrnmlPx0hqBk`**: Define `image_size`.
4.  **Cell `007f6ace`**: Define the `predict_disease` function and `disease_info` dictionary.
5.  **Cell `6ec6db65`**: Define the `disease_name_mapping` dictionary.

Once these cells are executed, you can then proceed to the Gradio cell (`5f2e4086`). I will now modify the Gradio cell to use the previously saved `teeth_disease_transfer_model.h5`.

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image
import tensorflow as tf

# Assuming transfer_model, disease_info, and disease_name_mapping are already defined and available

# Load the transfer learning model (which was saved previously)
fine_tuned_model_path = 'teeth_disease_transfer_model.h5'
try:
    fine_tuned_model = tf.keras.models.load_model(fine_tuned_model_path)
    print(f"Transfer learning model loaded successfully from {fine_tuned_model_path}")
except Exception as e:
    print(f"Error loading transfer learning model: {e}")
    fine_tuned_model = None # Handle case where model loading fails


def predict_and_display_info(img):
    if fine_tuned_model is None:
        print("Error: Model not loaded in predict function.") # Log error
        return ("Error: Model not loaded.", "", "", "", "") # Return error if model loading failed

    # Save the Gradio image to a temporary file to use with the predict_disease function
    img_path = "temp_image.png"
    img.save(img_path)
    print(f"Saved image to {img_path}") # Log

    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(128, 128)) # Ensure this matches your model's input size
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Create a batch
    img_array = img_array / 255.0  # Normalize the image

    print(f"Image array shape after preprocessing: {img_array.shape}") # Log

    # Make a prediction using the transfer learning model
    predictions = fine_tuned_model.predict(img_array)
    print(f"Model raw output (likely probabilities due to softmax activation): {predictions}") # Log raw predictions
    # The model's last layer has softmax activation, so predictions[0] is already the probability distribution
    score = predictions[0]
    print(f"Probability scores: {score}") # Log probability scores

    # Get the predicted class index
    predicted_class_index = np.argmax(score)
    print(f"Predicted class index: {predicted_class_index}") # Log

    # Get the predicted class name
    # Make sure class_names is available (define it globally or pass it)
    try:
        predicted_class_name_short = class_names[predicted_class_index]
        print(f"Predicted class name (short): {predicted_class_name_short}") # Log
    except NameError:
        print("Error: class_names not defined in predict function.") # Log error
        return ("Error: class_names not defined.", "", "", "", "")


    # Get the full disease name using the mapping, or use the short name if not found
    # Make sure disease_name_mapping is available (define it globally or pass it)
    try:
        predicted_class_name_full = disease_name_mapping.get(predicted_class_name_short, predicted_class_name_short)
        print(f"Predicted class name (full): {predicted_class_name_full}") # Log
    except NameError:
         predicted_class_name_full = predicted_class_name_short # Use short name if mapping not available
         print("Warning: disease_name_mapping not defined in predict function. Using short name.") # Log warning


    # Retrieve the additional information using the short name as the key
    # Make sure disease_info is available (define it globally or pass it)
    try:
        info = disease_info.get(predicted_class_name_short, {
            'precautions': ['Information not available.'],
            'warnings': ['Information not available.'],
            'additional_info': 'Information not available.'
        })
        print(f"Retrieved info for class: {predicted_class_name_short}") # Log
    except NameError:
        info = {
            'precautions': ['Information not available (disease_info not defined).'],
            'warnings': ['Information not available (disease_info not defined).'],
            'additional_info': 'Information not available (disease_info not defined).'
        }
        print("Warning: disease_info not defined in predict function. Using default info.") # Log warning


    final_confidence = score[predicted_class_index] # score is already a NumPy array
    print(f"Final calculated confidence: {final_confidence:.2f}") # Log final confidence


    precautions = "\n- " + "\n- ".join(info['precautions'])
    warnings = "\n- " + "\n- ".join(info['warnings'])
    additional_info = info['additional_info']

    return (f"Predicted Disease: {predicted_class_name_full}", # Display full name
            f"Confidence: {final_confidence:.2f}", # Format the confidence of the predicted class
            f"Precautions: {precautions}",
            f"Warnings: {warnings}",
            f"Additional Info: {additional_info}")


# Create the Gradio interface
iface = gr.Interface(
    fn=predict_and_display_info,
    inputs=gr.Image(type="pil", label="Upload an image of a tooth/mouth"),
    outputs=[
        gr.Textbox(label="Predicted Disease"),
        gr.Textbox(label="Confidence"),
        gr.Textbox(label="Precautions"),
        gr.Textbox(label="Warnings"),
        gr.Textbox(label="Additional Information")
    ],
    title="Teeth Disease Predictor",
    description="Upload an image to get a disease prediction and related information.",
    theme="gradio/ceres" # Added theme
)

# Launch the interface
iface.launch(debug=True)

Transfer learning model loaded successfully from teeth_disease_transfer_model.h5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1069: UserWarning: Cannot load gradio/ceres. Caught Exception: The space gradio/ceres does not exist
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d0af6ae1eae9c31257.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Saved image to temp_image.png
Image array shape after preprocessing: (1, 128, 128, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step
Model raw output (likely probabilities due to softmax activation): [[5.2844716e-04 1.5884507e-04 4.5375031e-04 1.4748907e-02 2.2549969e-03
  9.8038542e-01 1.4695987e-03]]
Probability scores: [5.2844716e-04 1.5884507e-04 4.5375031e-04 1.4748907e-02 2.2549969e-03
 9.8038542e-01 1.4695987e-03]
Predicted class index: 5
Predicted class name (short): OLP
Predicted class name (full): Oral Lichen Planus
Retrieved info for class: OLP
Final calculated confidence: 0.98
Saved image to temp_image.png
Image array shape after preprocessing: (1, 128, 128, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
Model raw output (likely probabilities due to softmax activation): [[0.01798922 0.02453931 0.00324633 0.01682802 0.10761747 0.7978291
  0.03195061]]
Probability scores: [0.01798922 0.02453931 0.00324633 0.01682802 0.10761747 0.7978291
 0.03195061]
Predicted class index: 5
Predicted clas

Finally, save the fine-tuned model.

In [ ]:
# Save the fine-tuned model
transfer_model.save('teeth_disease_fine_tuned_model.h5')

print("Fine-tuned model saved as 'teeth_disease_fine_tuned_model.h5'")


Fine-tuned model saved as 'teeth_disease_fine_tuned_model.h5'


In [ ]:
%pip install gradio

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image
import tensorflow as tf
import os # Import os module to check for file existence

# Assuming disease_info and disease_name_mapping are already defined and available
# You MUST run cells 8warN7Aobre_, 6c89ed5a, SrnmlPx0hqBk, 007f6ace, 6ec6db65 before running this cell.

# Path to the initially transfer-learned model (which was saved)
model_to_load_path = 'teeth_disease_transfer_model.h5'

fine_tuned_model = None # Initialize to None

# Check if the model file exists
if not os.path.exists(model_to_load_path):
    print(f"Error: Model file '{model_to_load_path}' not found.")
    print("To get the Gradio interface running, you must first ensure this model is saved.")
    print("Please re-run the following cells in order to generate and save this model:")
    print("1. Cell 'e34b43d3' (Load VGG16 base model)")
    print("2. Cell '6c839607' (Build transfer model)")
    print("3. Cell '1e7191a7' (Compile transfer model)")
    print("4. Cell '6d8c6f1c' (Train initial transfer model for 10 epochs)")
    print("5. Cell 'aeab26fc' (Save 'teeth_disease_transfer_model.h5')")
    print("After these steps, you can run this Gradio cell again.")
else:
    try:
        fine_tuned_model = tf.keras.models.load_model(model_to_load_path)
        print(f"Transfer learning model loaded successfully from {model_to_load_path}")
    except Exception as e:
        print(f"Error loading model from {model_to_load_path}: {e}")

def predict_and_display_info(img):
    if fine_tuned_model is None:
        return ("Error: Model not loaded. Please check the console for instructions.", "", "", "", "")

    # Save the Gradio image to a temporary file
    img_path = "temp_image.png"
    img.save(img_path)

    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0

    predictions = fine_tuned_model.predict(img_array)
    score = predictions[0]
    predicted_class_index = np.argmax(score)

    # Check for presence of global variables
    if 'class_names' not in globals():
        return ("Error: 'class_names' not defined. Please run essential setup cells.", "", "", "", "")
    predicted_class_name_short = class_names[predicted_class_index]

    if 'disease_name_mapping' not in globals():
        predicted_class_name_full = predicted_class_name_short
    else:
        predicted_class_name_full = disease_name_mapping.get(predicted_class_name_short, predicted_class_name_short)

    if 'disease_info' not in globals():
        info = {
            'precautions': ['Information not available (disease_info not defined).'],
            'warnings': ['Information not available (disease_info not defined).'],
            'additional_info': 'Information not available (disease_info not defined).'
        }
    else:
        info = disease_info.get(predicted_class_name_short, {
            'precautions': ['Information not available.'],
            'warnings': ['Information not available.'],
            'additional_info': 'Information not available.'
        })

    final_confidence = score[predicted_class_index]
    precautions = "\n- " + "\n- ".join(info['precautions'])
    warnings = "\n- " + "\n- ".join(info['warnings'])
    additional_info = info['additional_info']

    return (f"Predicted Disease: {predicted_class_name_full}",
            f"Confidence: {final_confidence:.2f}",
            f"Precautions: {precautions}",
            f"Warnings: {warnings}",
            f"Additional Info: {additional_info}")

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_and_display_info,
    inputs=gr.Image(type="pil", label="Upload an image of a tooth/mouth"),
    outputs=[
        gr.Textbox(label="Predicted Disease"),
        gr.Textbox(label="Confidence"),
        gr.Textbox(label="Precautions"),
        gr.Textbox(label="Warnings"),
        gr.Textbox(label="Additional Information")
    ],
    title="Teeth Disease Predictor",
    description="Upload an image to get a disease prediction and related information.",
    theme="gradio/ceres"
)

# Launch the interface
iface.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1069: UserWarning: Cannot load gradio/ceres. Caught Exception: The space gradio/ceres does not exist
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Transfer learning model loaded successfully from teeth_disease_transfer_model.h5
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://830cb412fb10f2688a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://830cb412fb10f2688a.gradio.live


In [ ]:
import shutil
shutil.copy('teeth_disease_fine_tuned_model.h5', '/content/drive/MyDrive/teeth_disease_fine_tuned_model.h5')
print("Model saved to Google Drive.")

Model saved to Google Drive.


### Load the Test Dataset

In [ ]:
# Define the path to the test dataset
test_dir = '/content/drive/MyDrive/Teeth_Dataset/Testing'

# Import the necessary function (if not already imported)
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow as tf

# Load the test dataset
test_dataset = image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=image_size, # Use the same image size as training/validation
    batch_size=32,
    shuffle=False # No need to shuffle the test dataset
)

print("Test dataset loaded.")

Found 1028 files belonging to 7 classes.
Test dataset loaded.


### Preprocess the Test Dataset

In [ ]:
# Define a preprocessing function for normalization (same as training/validation)
def preprocess_image(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32) / 255.0
    return image, label

# Apply preprocessing to the test dataset
test_dataset = test_dataset.map(preprocess_image)

# Configure the dataset for performance (optional, but good practice)
AUTOTUNE = tf.data.AUTOTUNE
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)


print("Test dataset preprocessing complete.")

Test dataset preprocessing complete.


### Evaluate the Fine-tuned VGG16 Model on the Test Dataset

In [ ]:
# Assuming 'fine_tuned_model' is loaded and available

# Evaluate the fine-tuned VGG16 model on the test dataset
loss_test, accuracy_test = fine_tuned_model.evaluate(test_dataset)

print(f"Fine-tuned VGG16 Model Test Loss: {loss_test}")
print(f"Fine-tuned VGG16 Model Test Accuracy: {accuracy_test}")

33/33 ━━━━━━━━━━━━━━━━━━━━ 184s 6s/step - accuracy: 0.9148 - loss: 0.2773
Fine-tuned VGG16 Model Test Loss: 0.3803636431694031
Fine-tuned VGG16 Model Test Accuracy: 0.8764591217041016


Evaluating on the test set provides a final measure of your model's performance on unseen data from the same distribution as your training data.

In [ ]:
# Create a requirements.txt file
!pip freeze > requirements.txt

# Task
Rerun the cells to build, train, and save the transfer learning model (`e34b43d3`, `6c839607`, `1e7191a7`, `6d8c6f1c`, `aeab26fc`), then re-run the essential setup cells for Gradio (`8warN7Aobre_`, `6c89ed5a`, `SrnmlPx0hqBk`, `007f6ace`, `6ec6db65`), and finally launch the Gradio interface by running cell `5f2e4086` to confirm it loads the `teeth_disease_transfer_model.h5` and functions correctly.

## Re-run Initial Transfer Learning and Save Model

### Subtask:
The `teeth_disease_transfer_model.h5` file was not found during the last attempt to load it. You need to re-run the cells that build the transfer learning model, train it for the initial 10 epochs, and then save it. This involves re-running cells `e34b43d3`, `6c839607`, `1e7191a7`, `6d8c6f1c`, and `aeab26fc` in that order. This will ensure the `teeth_disease_transfer_model.h5` file exists and is accessible.


**Reasoning**:
The model file `teeth_disease_transfer_model.h5` was not found. As instructed, the first step to rectify this is to re-run cell `e34b43d3` to load the VGG16 base model, which is a prerequisite for building the transfer learning model.



In [ ]:
from tensorflow.keras.applications import VGG16

# Choose a pre-trained model (e.g., VGG16)
# We will load weights pre-trained on ImageNet, and exclude the top classification layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

print("VGG16 pre-trained model loaded.")

VGG16 pre-trained model loaded.


**Reasoning**:
Following the instructions, the next step is to build the transfer learning model by adding new classification layers on top of the loaded VGG16 base model. This involves freezing the `base_model` layers and appending `Flatten`, `Dense`, and `Dropout` layers, then creating the `transfer_model`.



In [ ]:
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model

# Freeze the layers of the base model so they are not trained
base_model.trainable = False

# Add new classification layers on top of the base model
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

# Create the new model
transfer_model = Model(inputs=base_model.input, outputs=predictions)

transfer_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,764,295 (60.14 MB)

 Trainable params: 1,049,607 (4.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

**Reasoning**:
The previous attempt to build the transfer learning model failed because `num_classes` was not defined. To resolve this, I need to ensure that the dataset loading cell (`8warN7Aobre_`) is executed to get `class_names`, and then the cell that defines `num_classes` from `class_names` (`3462a44b`) is executed. After these dependencies are met, the current step to build the transfer model can be re-attempted.



In [ ]:
import os

# Update the paths to the correct dataset location
train_dir = '/content/drive/MyDrive/Teeth_Dataset/Training'
validation_dir = '/content/drive/MyDrive/Teeth_Dataset/Validation'

# Import the necessary function
from tensorflow.keras.utils import image_dataset_from_directory

# Load the training dataset
train_dataset = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(128, 128),
    batch_size=32
)

# Load the validation dataset
val_dataset = image_dataset_from_directory(
    validation_dir,
    labels='inferred',
    label_mode='int',
    image_size=(128, 128),
    batch_size=32
)

# Get the class names
class_names = train_dataset.class_names
print(f"Class names: {class_names}")

Found 3087 files belonging to 7 classes.
Found 1028 files belonging to 7 classes.
Class names: ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']


In [ ]:
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

Number of classes: 7


**Reasoning**:
The previous execution failed with a `NotFoundError`, indicating the system could not find the specified dataset directories on Google Drive. Although `drive.mount` was run previously, sometimes re-mounting can resolve transient access issues. Therefore, the next step is to re-execute the Google Drive mount command to ensure the drive is properly accessible, which is a prerequisite for loading the dataset.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task
Executing the following cells to rebuild the transfer learning model: `24e9f1b7`. This step will define the `transfer_model` with the correct number of output classes using the pre-trained VGG16 base.

Next, I will compile the transfer learning model by executing cell `79fa26ff` and then proceed to train it using cell `6d8c6f1c`. Once trained, I will save the model in cell `aeab26fc`. After that, I will ensure all necessary Gradio variables are defined by executing cells `8warN7Aobre_`, `6c89ed5a`, `SrnmlPx0hqBk`, `007f6ace`, and `6ec6db65`, before finally launching the Gradio interface with cell `5f2e4086`.

## Rebuild Transfer Learning Model

### Subtask:
Re-run cell '24e9f1b7' to build the transfer learning model. This uses the VGG16 base model and adds new classification layers on top, now that num_classes is correctly defined.


**Reasoning**:
The user explicitly asked to re-run cell `24e9f1b7` to build the transfer learning model now that `num_classes` is correctly defined.

